# Part 1 — Step 4: Model Evaluation

Compares YOLOv8 and Faster R-CNN on the **held-out test set**.

**Metrics:**
| Metric | Meaning |
|--------|---------|
| mAP@50 | Mean Average Precision at IoU ≥ 0.50 — standard detection benchmark |
| mAP@50-95 | mAP averaged over IoU thresholds 0.50–0.95 — stricter COCO metric |
| Precision | Of all predicted boxes, what fraction are correct |
| Recall | Of all ground-truth boxes, what fraction were found |

**Prerequisites:** Run notebooks 02 and 03 first to produce checkpoints.

In [ ]:
# Fix working directory so relative paths work in both Colab and locally
import os
from pathlib import Path

notebook_dir = Path('part1_detection')
if Path('/content').exists():  # we're in Colab
    os.chdir('/content/AcneDetection/part1_detection')
else:
    # Locally: run from repo root, adjust to notebook dir
    if Path('part1_detection').exists():
        os.chdir('part1_detection')

print(f'Working directory: {os.getcwd()}')

In [ ]:
import json
from pathlib import Path

import torch
from PIL import Image
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO

In [ ]:
DATA_DIR    = Path("../data/acne04")
YOLO_YAML   = Path("../data/acne04_yolo/dataset.yaml")
YOLO_CKPT   = Path("../outputs/yolov8/acne04/weights/best.pt")
FRCNN_CKPT  = Path("../outputs/faster_rcnn/best.pth")
OUT_FILE    = Path("../outputs/evaluation_results.json")
NUM_CLASSES = 5
CONF        = 0.25

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Evaluate YOLOv8

YOLOv8's built-in `val()` computes all metrics automatically.

In [ ]:
yolo_model   = YOLO(str(YOLO_CKPT))
yolo_metrics = yolo_model.val(data=str(YOLO_YAML), split="test", conf=CONF, verbose=False)

yolo_results = {
    "mAP50":     float(yolo_metrics.box.map50),
    "mAP50_95":  float(yolo_metrics.box.map),
    "precision": float(yolo_metrics.box.mp),
    "recall":    float(yolo_metrics.box.mr),
}
print("YOLOv8 results:")
for k, v in yolo_results.items():
    print(f"  {k:<12}: {v:.4f}")

## 2. Evaluate Faster R-CNN

Run inference on the test set, then score with `pycocotools COCOeval`.

In [ ]:
# Load model
frcnn = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
frcnn.roi_heads.box_predictor = FastRCNNPredictor(
    frcnn.roi_heads.box_predictor.cls_score.in_features, NUM_CLASSES)
frcnn.load_state_dict(torch.load(str(FRCNN_CKPT), map_location=device))
frcnn.to(device).eval()

# Load category mapping
ann_path = DATA_DIR / "test" / "_annotations.coco.json"
coco_gt  = COCO(str(ann_path))
with open(ann_path) as f:
    coco_data = json.load(f)
cats = sorted(coco_data["categories"], key=lambda c: c["id"])
label_to_cat = {i+1: c["id"] for i, c in enumerate(cats)}

# Inference
transform   = transforms.ToTensor()
coco_preds  = []
for img_id, img_meta in coco_gt.imgs.items():
    img    = Image.open(DATA_DIR / "test" / img_meta["file_name"]).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = frcnn(tensor)[0]
    for box, lbl, score in zip(out["boxes"], out["labels"], out["scores"]):
        if score.item() < CONF: continue
        x1, y1, x2, y2 = box.tolist()
        coco_preds.append({
            "image_id":    img_id,
            "category_id": label_to_cat.get(lbl.item(), lbl.item()),
            "bbox":        [x1, y1, x2-x1, y2-y1],
            "score":       score.item(),
        })
print(f"Total predictions: {len(coco_preds)}")

In [ ]:
coco_dt   = coco_gt.loadRes(coco_preds) if coco_preds else coco_gt.loadRes([])
evaluator = COCOeval(coco_gt, coco_dt, "bbox")
evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

frcnn_results = {
    "mAP50":    float(evaluator.stats[1]),
    "mAP50_95": float(evaluator.stats[0]),
    "precision": None,
    "recall":   float(evaluator.stats[6]),
}
print("\nFaster R-CNN results:")
for k, v in frcnn_results.items():
    print(f"  {k:<12}: {v:.4f}" if v is not None else f"  {k:<12}: —")

## 3. Comparison table

In [ ]:
import pandas as pd

rows = [
    {"Model": "YOLOv8s",      **yolo_results},
    {"Model": "Faster R-CNN", **frcnn_results},
]
df = pd.DataFrame(rows).set_index("Model")
df.columns = ["mAP@50", "mAP@50-95", "Precision", "Recall"]
display(df.style.format("{:.4f}", na_rep="—").highlight_max(axis=0, color="#d4edda"))

In [ ]:
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_FILE, "w") as f:
    json.dump({"yolov8": yolo_results, "faster_rcnn": frcnn_results}, f, indent=2)
print(f"Results saved → {OUT_FILE}")